In [1]:
from langchain_community.tools import DuckDuckGoSearchResults


In [6]:
search = DuckDuckGoSearchResults(backend="news", num_results=10)
search_web = DuckDuckGoSearchResults(backend="text", num_results=5)
search_results = search_web.invoke("site:pulse.zerodha.com trending market news India")

In [7]:
search_results

"snippet: Indian markets concluded the week on a strong note, continuing their upward trend. Buying interest in banking, auto, and consumer stocks fueled the gains. The Nifty50 and BSE Sensex saw significant advances. Volatility eased considerably. Experts provided outlooks for Nifty and Bank Nifty, along with sector-specific strategies for the upcoming week., title: Pulse by Zerodha - Latest financial and market news from all major Indian news sources aggregated in one place - Pulse, link: https://pulse.zerodha.com/, snippet: India's IPOmarketfaces potential disruptions as several companies risk missing their listing deadlines due to amarketdownturn. Firms like Credila Financial Services and Dorf-Ketal Chemicals are among those with approvals nearing expiry., title: Latest financial andmarketnewsfrom all majorIndiannewssources..., link: https://pulse.zerodha.com/?ref=stockmarketresources.info, snippet: Indianstockmarketswill trade for only three days this week. This shortened trading 

In [9]:
SEARCH_QUERIES = [
    "Nifty 50 Sensex BSE NSE market today India",
    "NSE BSE company deal contract merger acquisition India today",
    "Indian company quarterly results earnings profit loss today",
    "SEBI penalty regulation stock market India today",
    "RBI policy inflation impact banks India today",
    "Global crude oil US Fed dollar rupee market impact India today",
    "Ministry of defense railways major contract awarded today India",
    "Indian IT banking pharma auto energy stocks news today", # Combines 5 queries into 1
    "site:pulse.zerodha.com trending market news India"
]

In [11]:
search_news = DuckDuckGoSearchResults(backend="news", num_results=5)
search_web = DuckDuckGoSearchResults(backend="text", num_results=5) # 'text' is standard web search
import time
def fetch_ddg_news():
    """
    Uses DuckDuckGo to fetch news, handling News and Web backends appropriately.
    """
    all_results = []
    
    for query in SEARCH_QUERIES:
        try:
            # If the query is looking for a specific website (like Pulse), use the Web backend
            if "site:" in query:
                result = search_web.invoke(query)
            # Otherwise, use the standard News backend
            else:
                result = search_news.invoke(query)
                
            if result:
                all_results.append(result)
                
        except Exception as e:
            # Catching the exact DecodeError so it doesn't crash your whole script
            print(f"Warning: DuckDuckGo failed for query '{query}': {e}")
            
        # Add a 2-second sleep between searches to prevent DuckDuckGo from IP banning your bot
        time.sleep(2) 
        
    return "\n".join(all_results)
res=fetch_ddg_news()
print(res)

snippet: In early trade, market breadth was positive, with 2,033 stocks advancing against only 581 stocks declining on the NSE. 62 ..., title: Market Opening Bell: Sensex drops 243.57 points, Nifty holds 23,900 as crude oil prices inch up, link: https://www.indiatvnews.com/business/markets/stock-market-sensex-gift-nifty-today-sgx-asian-markets-global-market-nikkei-225-kospi-tata-steel-ntpc-share-in-focus-tcs-q4-results-2026-04-09-1036859, date: 2026-04-09T06:24:25+00:00, source: India TV News, snippet: Sensex, Stock Market Highlights: Both Sensex and Nifty opened negative on Tuesday after US, Iran failed to reach a ceasefire ..., title: Stock Market, Sensex Today: Sensex, Nifty Close In Green After Volatile Trading Session, link: https://www.ndtv.com/india-news/share-market-live-updates-markets-bse-sensex-nse-nifty50-gift-nifty-us-iran-ceasefire-stock-market-today-april-7-11320909, date: 2026-04-08T06:24:25+00:00, source: NDTV, snippet: Indian stock markets are closed today for Shri Ma

In [12]:
import os
import time, re
import schedule
import requests
import feedparser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults
from dotenv import load_dotenv
from datetime import datetime, timedelta

# ==========================================
# 1. SETUP DATES & ENVIRONMENT
# ==========================================
today = datetime.now()
yesterday = today - timedelta(days=1)

TODAY_STR = today.strftime('%Y-%m-%d')
YESTERDAY_STR = yesterday.strftime('%Y-%m-%d')

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")  
FINNHUB_KEY = os.getenv("FINNHUB_KEY")

print(f"Loaded config: Gemini key {'found' if GEMINI_API_KEY else 'missing'}, Telegram token {'found' if TELEGRAM_BOT_TOKEN else 'missing'}, Chat ID {'found' if TELEGRAM_CHAT_ID else 'missing'}, NewsAPI key {'found' if NEWSAPI_KEY else 'missing'}, Finnhub key {'found' if FINNHUB_KEY else 'missing'}")

# ==========================================
# 2. SETUP LLM & SEARCH TOOLS
# ==========================================
# Using 1.5-flash to take advantage of the 1,500 requests/day free tier limit
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    api_key=GEMINI_API_KEY
)


Loaded config: Gemini key found, Telegram token found, Chat ID found, NewsAPI key found, Finnhub key found
